In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# LLM 설정 (vLLM OpenAI-compatible)
from langchain_openai import ChatOpenAI

# Embedding 설정
from langchain_huggingface import HuggingFaceEmbeddings

# RAGAS
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

import os, json

In [2]:
# ============================================
# 1. LLM 설정 (vLLM 서버)
# ============================================
VLLM_BASE_URL = "http://10.10.100.45:8003/v1"
VLLM_MODEL_NAME = "openai/gpt-oss-20b"
temperature = 0.7
max_token = 30000

# OpenAI-compatible API 사용
llm = ChatOpenAI(
    base_url=VLLM_BASE_URL,
    model=VLLM_MODEL_NAME,
    api_key="not-needed",  # vLLM은 API 키 불필요
    temperature=temperature,
    max_tokens=max_token,
    timeout=600,  # ← 추가 (10분)
)

# RAGAS용 LLM wrapper
evaluator_llm = LangchainLLMWrapper(llm)

print(f"LLM 설정 완료: {VLLM_MODEL_NAME}")

LLM 설정 완료: openai/gpt-oss-20b


In [3]:
# ============================================
# 2. Embedding 설정 (HuggingFace BGE)
# ============================================
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
EMBEDDING_DEVICE = "cuda"  # 또는 "cuda"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": EMBEDDING_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

# RAGAS용 Embedding wrapper
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

print(f"Embedding 설정 완료: {EMBEDDING_MODEL}")

Embedding 설정 완료: BAAI/bge-base-en-v1.5


In [4]:
# ============================================
# 3. RAGAS 메트릭 설정
# ============================================
# Golden set 없이 사용 가능한 메트릭들

metrics = [
    # 답변이 context에 기반하는지 (hallucination 체크)
    Faithfulness(llm=evaluator_llm),
    
    # 답변이 질문에 관련되는지
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    
    # 검색된 context가 질문에 관련되는지 (reference 없이)
    LLMContextPrecisionWithoutReference(llm=evaluator_llm),
]

print("평가 메트릭:")
for m in metrics:
    print(f"  - {m.name}")

평가 메트릭:
  - faithfulness
  - answer_relevancy
  - llm_context_precision_without_reference


In [22]:
qna_root_path = '/home/kitemo/main/jupyter/2601/PE_Agent_Golden_QnA'
result_list = []

for json_file in os.listdir(qna_root_path):
    json_path = os.path.join(qna_root_path, json_file)

    with open(json_path, 'r', encoding='utf-8') as file:
        # json.load() 함수를 사용하여 파일 객체에서 데이터를 로드합니다.
        qna_result = json.load(file)

    selected_result_dict = {}
    selected_result_dict['query'] = qna_result['query']
    selected_result_dict['answer'] = qna_result['answer']
    selected_result_dict['retrieved_list'] = [result_dict['snippet'] for result_dict in qna_result['retrieved_docs']]

    result_list.append(selected_result_dict)

ragas_samples = [SingleTurnSample(
        user_input=result['query'],
        retrieved_contexts=result['retrieved_list'],
        response=result['answer'],
    ) for result in result_list]

In [23]:
# ============================================
# 5. 평가 실행
# ============================================
dataset = EvaluationDataset(samples=ragas_samples)

results = evaluate(
    dataset=dataset,
    metrics=metrics,
)

print("평가 완료!")
print(results)

Evaluating:   0%|          | 0/45 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[0]: TimeoutError()


평가 완료!
{'faithfulness': 0.3595, 'answer_relevancy': 0.7934, 'llm_context_precision_without_reference': 0.5778}


In [28]:
# ============================================
# 6. 결과 분석
# ============================================
import pandas as pd

# DataFrame으로 변환
df = results.to_pandas()
display(df)

# 메트릭별 평균
print("\n=== 메트릭별 평균 ===")
for col in df.columns:
    if col not in ["user_input", "retrieved_contexts", "response"]:
        print(f"{col}: {df[col].mean():.3f}")

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy,llm_context_precision_without_reference
0,ECOLITE3000 설비에서 PM Chamber 내부 View Port 쪽에 Lo...,"[-. 설비 Main Arr, LP2 Manual 상태\n# PM3\n-. Temp...",**원인** \nECOLITE3000 설비에서 PM Chamber 내부 View ...,NaN,0.869378,0.500000
1,SUPRA V에서 SUPRA Np로 개조된 설비의 설비 호기명은 무엇인가?,[The review confirmed that the existing portab...,제가 확인한 RAG 데이터에서는 **SUPRA V에서 SUPRA Np로 개조된 설비...,0.000000,0.775888,0.000000
2,SUPRA Np에서 발생한 Issue 내용들을 GCB 기반으로 정리해줄 수 있을까?,"[- 현장에서 Cable 정리할 수 있는 부분은 현장에서 정리 진행, -. 2.20...",**원인 / 진단 / 조치**\n\n| 항목 | 내용 |\n|------|-----...,0.583333,0.507848,1.000000
3,SUPRA III 설비에서 APC Pressure Hunting 발생 시 점검해야 ...,[-. PM Main Process 구간에서 300mTorr 로 유지해야 하는 구간...,**원인 / 진단 / 조치**\n\n| 단계 | 내용 | 근거 |\n|------|...,0.300000,0.774489,1.000000
4,SUPRA N TM ROBOT ENDEFFECTOR 장착 시 Screw 체결 토크 ...,"[-. S/W SETUP.INI 설정 ERROR, -. Setup.ini 설정 및 ...",1. **SUPRA N TM 로봇 엔드에펙터 장착 시 Screws의 토크 스펙을 R...,0.666667,0.843062,0.333333
5,INTEGER Plus 설비에서 PM PIN Motor 교체 시 Pin 높이는 몇으...,[-. 명일 탭작업 이후 작업 예정\n* JEL 김종연 부장님에 의하면 구형 로봇 ...,**답변 요약**\n\n1. **Pin 높이**: 15.3 / 15.3 / 15.3...,0.333333,0.782583,0.500000
6,GENEVA XP 설비에서 APC 관련 내용이 있는 GCB 번호를 알 수 있을까?,[- 현재 장착된 Bracket이 움직일 수 있는 공간이나 여유가 없어서 Senso...,해당 REFS에는 APC(공정 제어 장치)와 관련된 내용이 포함되어 있지 않습니다....,0.600000,0.926022,0.500000
7,SUPRA N Baffle 장착 시 Screw 체결 토크 스펙은 얼마인가?,[-. Baffle & Focus 세정품교체\n-. 배플스크류&와셔교체\n-. 탑리...,**SUPRA N Baffle 장착 시 Screw 체결 토크 스펙**\n\n1. 4...,0.000000,0.847831,0.333333
8,EPAGQ03에서 Source Unready Alarm이 발생한 이력에 대해 정리해...,"[WABPEB03 PM1-1 SOURCE UNREADY ALARM, - EPAT02...",**원인** \n현재 제공된 RAG 데이터(REFS)에서는 EPAGQ03에서 So...,0.600000,0.859070,0.500000
9,mySERVICE 이력 중 SEC SRD 라인의 EPA404호기 LL 점검 이력을 ...,[-. 5/12 EPD Cal 이력 有\n-. 현상 확인 결과 (EPD Log 확인...,제공된 REFS 중에서는 SEC SRD 라인의 EPA404호기 LL 점검 이력과 관...,0.800000,0.748362,1.000000



=== 메트릭별 평균 ===
faithfulness: 0.360
answer_relevancy: 0.793
llm_context_precision_without_reference: 0.578


In [29]:
df.to_csv('./ragas_results.csv')